In [45]:
import folium
import pandas as pd

In [46]:
m = folium.Map(
    location=[25.04, 121.52],
    zoom_start=8,
    control_scale=True
)

m

In [47]:
from pathlib import Path
import shutil

from pathlib import Path

project_root = Path("C:/Users/omistaja/Desktop/AS Project")

photo_root = project_root / "data" / "photos"

image_files = (
    list(photo_root.rglob("*.jpg")) +
    list(photo_root.rglob("*.JPG")) +
    list(photo_root.rglob("*.jpeg")) +
    list(photo_root.rglob("*.JPEG")) +
    list(photo_root.rglob("*.png")) +
    list(photo_root.rglob("*.PNG"))
)

image_lookup = {
    image.stem.strip().lower(): image
    for image in image_files
}

print("Images found:", len(image_lookup))

published_image_folder = Path("../data/photos")
published_image_folder.mkdir(parents=True, exist_ok=True)

filename_from_table = str(row["sign_photo_filename"]).strip()
lookup_key = Path(filename_from_table).stem.lower()

source_image = image_lookup.get(lookup_key)

if source_image is not None:

    published_filename = source_image.name

    destination = published_image_folder / published_filename

    if not destination.exists():
        shutil.copy2(source_image, destination)

    image_html = f"""
    <img
        src="images/{published_filename}"
        style="
            width:240px;
            max-height:180px;
            object-fit:contain;
            display:block;
            margin-bottom:8px;
        "
    >
    """
else:
    image_html = "<i>Image not available</i>"

Images found: 1044


In [53]:
excel_path = (
    "C:/Users/omistaja/Desktop/AS Project/Data/"
    "Spreadsheets/street_signs_taipei_sheets.xlsx"
)

signs = pd.read_excel(
    excel_path,
    sheet_name="OBSERVATION"
)

sign_layer = folium.FeatureGroup(
    name="Signs"
)

for _, row in signs.iterrows():

    filename = row["sign_photo_filename"]
    image_file = image_lookup.get(filename)

    popup_html = f"""
    <b>{row['sign_id']}</b><br>
    Text: {row['sign_content']}
    Last observed: {row['observation_date']}<br>
    Status: {row['sign_status']}
    """

    folium.CircleMarker(
        location=[row["sign_latitude"], row["sign_longitude"]],
        radius=4,
        tooltip=row["sign_id"],
        popup=popup_html,
        fill=True,
        fill_opacity=0.8,
        weight=1
    ).add_to(sign_layer)

sign_layer.add_to(m)

m
# m.save('interactive_map.html')